In [8]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, avg, round, lit, count, stddev

# 1. Alustetaan sessio ja ladataan MongoDB-ajuri
spark = SparkSession.builder \
    .appName("PISA-Reading-Analysis") \
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:10.4.0") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

In [11]:
def get_finland_stats(year):
    uri = f"mongodb://127.0.0.1:27017/pisa_database.year{year}"
    df = spark.read.format("mongodb").option("spark.mongodb.read.connection.uri", uri).load()
    
    fin_df = df.filter(col("CNT") == "FIN")
    
    # Choosing the correct ICT column based on the year
    ict_col_name = "ICTAVHOM_SUM" if "ICTAVHOM_SUM" in df.columns else "ICTAVHOM"
    
    # Averages and rounding directly in Spark
    stats = fin_df.select(
        round(avg("PV1READ"), 2).alias("avg_reading"),
        round(avg(ict_col_name), 2).alias("avg_ict")
    ).collect()[0]
    
    return stats

try:
    print("\n--- PISA ANALYSIS READING: FINLAND 2012 vs 2022 ---\n")
    
    stats_2012 = get_finland_stats(2012)
    stats_2022 = get_finland_stats(2022)
    
    print(f"Year 2012 (Reading): {stats_2012['avg_reading']} points")
    print(f"Year 2012 (ICT Availability): {stats_2012['avg_ict']} (WLE index)")
    print("-" * 40)
    print(f"Year 2022 (Reading): {stats_2022['avg_reading']} points")
    print(f"Year 2022 (ICT Availability): {stats_2022['avg_ict']} (sum variable)")
    
    # Calculating change
    change = stats_2022['avg_reading'] - stats_2012['avg_reading']
    print(f"\nChange in reading proficiency: {change:.2f} points.")

except Exception as e:
    print(f"Error in analysis: {e}")


--- PISA ANALYSIS READING: FINLAND 2012 vs 2022 ---

Year 2012 (Reading): 510.91 points
Year 2012 (ICT Availability): 38.49 (WLE index)
----------------------------------------
Year 2022 (Reading): 477.98 points
Year 2022 (ICT Availability): 5.67 (sum variable)

Change in reading proficiency: -32.93 points.


In [12]:
def get_correlation(year):
    uri = f"mongodb://127.0.0.1:27017/pisa_database.year{year}"
    df = spark.read.format("mongodb").option("spark.mongodb.read.connection.uri", uri).load()
    
    # 1. Valitaan oikea sarake dynaamisesti
    # Tarkistetaan löytyykö _SUM versio, muuten käytetään perusversiota
    if "ICTAVHOM_SUM" in df.columns:
        ict_col = "ICTAVHOM_SUM"
    elif "ICTAVHOM" in df.columns:
        ict_col = "ICTAVHOM"
    else:
        print(f"Varoitus: Vuoden {year} datasta ei löytynyt ICTAVHOM-saraketta.")
        return None

    # 2. Suodatetaan Suomi ja varmistetaan, ettei valitussa sarakkeessa ole tyhjiä
    # Huom: Käytetään col(ict_col) muuttujaa merkkijonon sijasta
    fin_df = df.filter(
        (col("CNT") == "FIN") & 
        (col(ict_col).isNotNull()) & 
        (col("PV1MATH").isNotNull())
    )
    
    # 3. Lasketaan korrelaatio dynaamisesti valitulla sarakkeella
    try:
        correlation = fin_df.stat.corr(ict_col, "PV1MATH")
        return correlation
    except Exception as e:
        print(f"Virhe korrelaation laskennassa vuodelle {year}: {e}")
        return None

corr_2012 = get_correlation(2012)
corr_2022 = get_correlation(2022)

print(f"\n--- CORRELATION ANALYSIS (ICT vs. MATHEMATICS) ---")
print(f"Correlation in 2012: {corr_2012:.3f}")
print(f"Correlation in 2022: {corr_2022:.3f}")


--- CORRELATION ANALYSIS (ICT vs. MATHEMATICS) ---
Correlation in 2012: 0.067
Correlation in 2022: 0.117


In [6]:
def analyze_reading_by_time(year):
    uri = f"mongodb://127.0.0.1:27017/pisa_database.year{year}"
    df = spark.read.format("mongodb").option("spark.mongodb.read.connection.uri", uri).load()
    
    if year == 2012:
        # 2012 käyttää raakoja kategorioita (1-7)
        df = df.withColumn("Usage_Group", 
            when(col("ICTWKDY").isin(1, 2, 3), "1. Low (0-1h)")
            .when(col("ICTWKDY").isin(4, 5), "2. Moderate (1-4h)")
            .when(col("ICTWKDY").isin(6, 7), "3. Heavy (4h+)")
            .otherwise(None))
    else:
        # 2022 käyttää WLE-indeksiä (liukulukuja)
        # Säädetään rajat vastaamaan suurin piirtein samaa jakaumaa
        df = df.withColumn("Usage_Group", 
            when(col("ICTWKDY") < -0.5, "1. Low (Below Avg)")
            .when((col("ICTWKDY") >= -0.5) & (col("ICTWKDY") <= 1.0), "2. Moderate (Avg)")
            .when(col("ICTWKDY") > 1.0, "3. Heavy (High Use)")
            .otherwise(None))

    stats = df.filter((col("CNT") == "FIN") & (col("Usage_Group").isNotNull())) \
        .groupBy("Usage_Group") \
        .agg(
            round(avg("PV1READ"), 1).alias("Avg_Reading"),
            # Lisätään määrä, jotta nähdään ryhmien koko
            count("PV1READ").alias("N_Students") 
        ).orderBy("Usage_Group")
    
    return stats

# 3. Suoritetaan analyysi
try:
    print("\n--- LUKUTAITO VS. INTERNET-AIKA SUOMESSA ---")
    print("Vuosi 2012:")
    analyze_reading_by_time(2012).show()
    
    print("\nVuosi 2022:")
    analyze_reading_by_time(2022).show()

except Exception as e:
    print(f"Virhe analyysissä: {e}")


--- LUKUTAITO VS. INTERNET-AIKA SUOMESSA ---
Vuosi 2012:
+------------------+-----------+----------+
|       Usage_Group|Avg_Reading|N_Students|
+------------------+-----------+----------+
|     1. Low (0-1h)|      524.1|      1910|
|2. Moderate (1-4h)|      517.5|      5309|
|    3. Heavy (4h+)|      498.4|      1257|
+------------------+-----------+----------+


Vuosi 2022:
+-------------------+-----------+----------+
|        Usage_Group|Avg_Reading|N_Students|
+-------------------+-----------+----------+
| 1. Low (Below Avg)|      505.4|      1728|
|  2. Moderate (Avg)|      500.9|      5451|
|3. Heavy (High Use)|      430.7|      1056|
+-------------------+-----------+----------+



In [9]:
def check_reading_correlation(year):
    uri = f"mongodb://127.0.0.1:27017/pisa_database.year{year}"
    df = spark.read.format("mongodb").option("spark.mongodb.read.connection.uri", uri).load()
    
    # Suodatetaan Suomi ja poistetaan tyhjät (nullit sotkevat korrelaation)
    fin_df = df.filter(
        (col("CNT") == "FIN") & 
        (col("ICTWKDY").isNotNull()) & 
        (col("PV1READ").isNotNull())
    )
    
    # Lasketaan korrelaatio
    r = fin_df.stat.corr("ICTWKDY", "PV1READ")
    
    # Lasketaan myös hajonta, joka kertoo ryhmän sisäisistä eroista
    std_dev = fin_df.select(stddev("PV1READ")).collect()[0][0]
    
    return r, std_dev

try:
    r12, sd12 = check_reading_correlation(2012)
    r22, sd22 = check_reading_correlation(2022)
    
    print("\n--- LUKUTAIDON JA RUUTUAJAN KORRELAATIO ---")
    print(f"Vuosi 2012: r = {r12:.3f} (Hajonta: {sd12:.1f})")
    print(f"Vuosi 2022: r = {r22:.3f} (Hajonta: {sd22:.1f})")
    
    # Lasketaan korrelaation muutos prosentteina "selitysvoimasta" (r^2)
    print(f"\nAnalyysi: Korrelaatio on muuttunut {((r22-r12)/abs(r12))*100:.1f} %")

except Exception as e:
    print(f"Virhe: {e}")


--- LUKUTAIDON JA RUUTUAJAN KORRELAATIO ---
Vuosi 2012: r = -0.260 (Hajonta: 100.4)
Vuosi 2022: r = -0.195 (Hajonta: 103.7)

Analyysi: Korrelaatio on muuttunut 25.1 %
